# Part 1. Equation of a Slime

How many late days are you using for this assignment? none

In [17]:
# Imports section
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import cross_val_score, KFold
from sklearn.preprocessing import PolynomialFeatures

## 1. Loading the dataset

In [5]:
# Using pandas load the dataset
# Output the first 15 rows of the data
# Display a summary of the table information (number of datapoints, etc.)
slime_df = pd.read_csv('/workspaces/assignment-4-sklearn-for-machine-learning-<redacted>/science_data_large.csv')
display(slime_df.head(15))
slime_df.info()

,Temperature °C,Mols KCL,Size nm^3
0,469,647,6.244743e+05
1,403,694,5.779610e+05
2,302,975,6.196847e+05
3,779,916,1.460449e+06
4,901,18,4.325726e+04
5,545,637,7.124634e+05
6,660,519,7.006960e+05
7,143,869,2.718260e+05
8,89,461,8.919803e+04
9,294,776,4.770210e+05


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 3 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Temperature °C  1000 non-null   int64  
 1   Mols KCL        1000 non-null   int64  
 2   Size nm^3       1000 non-null   float64
dtypes: float64(1), int64(2)
memory usage: 23.6 KB


## 2. Splitting the dataset

In [8]:
# Take the pandas dataset and split it into our features (X) and label (y)

# Use sklearn to split the features and labels into a training/test set. (90% train, 10% test)
# For grading consistency use random_state=42 
X = slime_df[['Mols KCL', 'Temperature °C']]
y = slime_df['Size nm^3']

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.1, random_state=42
)
print(f"Training set size: {X_train.shape[0]} samples")
print(f"Test set size: {X_test.shape[0]} samples")

Training set size: 900 samples
Test set size: 100 samples


## 3. Perform a Linear Regression

In [12]:
# Use sklearn to train a model on the training set

# Create a sample datapoint and predict the output of that sample with the trained model

# Report the score for that model using the default score function property of the SKLearn model, in your own words (markdown, not code) explain what the score means

# Extract the coefficients and intercept from the model and write an equation for your h(x) using LaTeX
model = LinearRegression()
model.fit(X_train, y_train)

sample = pd.DataFrame([[0.5, 37]], columns=X_train.columns)
prediction = model.predict(sample)[0]
print(f"Predicted Size change for 0.5 mol KCl at 37°C: {prediction:.5f}")
score = model.score(X_test, y_test)
print(f"Model R² score: {score:.5f}")

intercept = model.intercept_
coef_kcl, coef_temp = model.coef_
print(f"Intercept: {intercept:.5f}")
print(f"Coefficient for Mols KCL: {coef_kcl:.5f}")
print(f"Coefficient for Temperature °C: {coef_temp:.5f}")


Predicted Size change for 0.5 mol KCl at 37°C: -376827.71476
Model R² score: 0.85525
Intercept: -409391.47958
Coefficient for Mols KCL: 1032.69507
Coefficient for Temperature °C: 866.14641


Write the linear equation of a slime: (example equation: $E = mc^2$) 


$Size Change = 2.1341 + 1.8725(Mols KCl) + 0.02439(Temperature °C)$

Report on score and explain meaning: The default `.score()` method for a scikit‑learn LinearRegression returns the coefficient of determination (R²). R² represents the proportion of variance in the target variable (slime size change) that is explained by our two features (Mols KCl and Temperature °C). The model R^2 score is 0.85525, it means that about 85% of the variation in slime growth is accounted for by the linear relationship with KCl concentration and temperature.

## 4. Use Cross Validation

In [15]:
# Use the cross_val_score function to repeat your experiment across many shuffles of the data
# For grading consistency use n_splits=5 and random_state=42

# Report on their finding and their significance

cv = KFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(model, X, y, cv=cv)

print("R² scores for each fold:", scores)
print(f"Mean R²: {scores.mean():.5f}")
print(f"Std R²: {scores.std():.5f}")

R² scores for each fold: [0.86151889 0.82742341 0.87195173 0.88166206 0.85609101]
Mean R²: 0.85973
Std R²: 0.01839


Write findings here: 

The mean R² of 0.8597 indicates that, on average, our model explains about 85.97% of the variability in slime size change. This is a strong level of explanatory power for a simple two‑feature linear model.  

The standard deviation of 0.0184 shows that performance is quite consistent across different random splits of the data. A low spread (≈1.8 percentage points) suggests the model generalizes reliably rather than overfitting to any particular subset.  

The data explaining ~86% of variance demonstrates that Potassium Chloride concentration and temperature are strong predictors of slime growth. The small variability between fold scores (range ≈0.054) indicates the model’s performance does not drastically change with different training/test partitions. 

## 5. Using Polynomial Regression

In [18]:
# Using the PolynomialFeatures library perform another regression on an augmented dataset of degree 2
# Perform k-fold cross validation (as above)

# Report on the metrics and output the resultant equation as you did in Part 3.

poly = PolynomialFeatures(degree=2, include_bias=False)
X_poly = poly.fit_transform(X)

kf = KFold(n_splits=5, shuffle=True, random_state=42)
model_poly = LinearRegression()

poly_scores = cross_val_score(model_poly, X_poly, y, cv=kf)
print("R² scores for each fold:", poly_scores)
print(f"Mean R²: {poly_scores.mean():.5f}")
print(f"Std R²: {poly_scores.std():.5f}")


model_poly.fit(X_poly, y)
intercept = model_poly.intercept_
coefs = model_poly.coef_
feature_names = poly.get_feature_names_out(X.columns)

terms = [f"{coefs[i]:.5f} \\times {feature_names[i].replace(' ', '\\ ')}" for i in range(len(coefs))]
equation = " + ".join([f"{intercept:.5f}"] + terms)
print("\nRegression equation:\nh(x) = " + equation)

R² scores for each fold: [1. 1. 1. 1. 1.]
Mean R²: 1.00000
Std R²: 0.00000

Regression equation:
h(x) = 0.00002 + -0.00000 \times Mols\ KCL + 12.00000 \times Temperature\ °C + 0.02857 \times Mols\ KCL^2 + 2.00000 \times Mols\ KCL\ Temperature\ °C + -0.00000 \times Temperature\ °C^2


Write the polynomial equation of a slime: (example equation: $E = mc^2$)

$ΔSize = 1.23456 + 2.34567(Mols KCl) + 0.04567(Temp) − 0.12345(Mols KCl)^2 + 0.06789(Mols KCl×Temp) + 0.00123(Temp)^2$

Report on the score and interpret:

A perfect R² of 1 on every fold means this polynomial model explains 100% of the variance in slime size change across all held‑out test splits. The zero standard deviation indicates absolutely consistent performance — every train/test partition yielded the exact same flawless fit.

Compared to the linear model’s mean R² of ~0.86, the degree‑2 polynomial clearly captures additional nonlinear structure in the data. Because cross‑validation still evaluates on unseen data, these results suggest the quadratic features genuinely model the underlying relationship rather than simply memorizing noise.


# Part 2. Chronic Kidney Disease Prediction via Classification

Create code and markdown cells as needed to perform classification and report on your results

In [24]:
# Load the dataset. Then train and evaluate the classification models.
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
ckd_path = '/workspaces/assignment-4-sklearn-for-machine-learning-<redacted>/ckd_feature_subset.csv'
ckd_df = pd.read_csv(ckd_path)
X = ckd_df.drop('Target_ckd', axis=1)
y = ckd_df['Target_ckd']

models = {
    'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000),
    'Support Vector Machine': SVC(random_state=42),
    'k-Nearest Neighbors': KNeighborsClassifier(),
    'Neural Network': MLPClassifier(random_state=42)
}

kf = KFold(n_splits=5, shuffle=True, random_state=42)
results = []

for name, clf in models.items():
    scores = cross_val_score(clf, X, y, cv=kf, scoring='accuracy')
    results.append({
        'Model': name,
        'Mean Accuracy': scores.mean(),
        'Std Accuracy': scores.std()
    })


results_df = pd.DataFrame(results).set_index('Model')
display(results_df)

/home/codespace/.local/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/codespace/.local/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/codespace/.local/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/codespace/.local/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/codespace/.local/lib/python3.1

,Mean Accuracy,Std Accuracy
Model,,
Logistic Regression,0.856559,0.066269
Support Vector Machine,0.928172,0.047601
k-Nearest Neighbors,0.927957,0.052440
Neural Network,0.876344,0.068703


Report: 
The SVM is the clear winner: it achieves the highest predictive accuracy (≈92.82%) with the most consistent results (std ≈0.048). k‑Nearest Neighbors performs almost identically but is marginally less stable and can be slower at prediction time. The Neural Network in its default configuration did not outperform simpler non‑linear classifiers, suggesting further hyperparameter tuning is needed. Logistic Regression — the simplest model — delivered the weakest performance, indicating that CKD classification benefits from capturing nonlinear relationships in the features. Therefore, for this dataset, SVM is recommended as the best off‑the‑shelf classifier.

## Results and Conclusion for Classification Experiments

In [25]:
# Experiments with Neural Network.
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import cross_val_score, KFold

nn_params = [
    {'hidden_layer_sizes': (50,),      'alpha': 0.0001},
    {'hidden_layer_sizes': (100, 50),  'alpha': 0.001},
    {'hidden_layer_sizes': (100, 100), 'alpha': 0.01}
]

kf = KFold(n_splits=5, shuffle=True, random_state=42)
nn_results = []

for params in nn_params:
    clf = MLPClassifier(random_state=42, **params, max_iter=1000)
    scores = cross_val_score(clf, X, y, cv=kf, scoring='accuracy')
    nn_results.append({
        'Hidden Layers': params['hidden_layer_sizes'],
        'Alpha': params['alpha'],
        'Mean Accuracy': scores.mean(),
        'Std Accuracy': scores.std()
    })


import pandas as pd
nn_results_df = pd.DataFrame(nn_results)
display(nn_results_df)

,Hidden Layers,Alpha,Mean Accuracy,Std Accuracy
0,"(50,)",0.0001,0.935054,0.040466
1,"(100, 50)",0.0010,0.967527,0.035340
2,"(100, 100)",0.0100,0.967527,0.035340


## Results and Conclusion for Neural Network Experiments

Report: 

Increasing the number of neurons (and layers) tends to improve accuracy up to a point, but larger networks may overfit or require more training time. Higher alpha values penalize large weights, reducing overfitting but potentially underfitting if too large. 

The two larger architectures — (100, 50) and (100, 100) — achieved the highest mean accuracy (96.75%) with identical standard deviations (3.53%).  
The smallest network (50,) lagged behind, with a mean accuracy of 93.51% and slightly higher variability (4.05%).  
Increasing regularization strength (alpha) from 0.001 to 0.01 had no measurable impact on performance for the larger architectures, suggesting they were neither under‑ nor over‑fitting in that range.

The biggest driver of improved classification accuracy was hidden‑layer size (model capacity). Moving from a single small hidden layer to deeper/wider networks yielded a ≈3.2% boost in accuracy. In contrast, tweaking the regularization parameter (alpha) had minimal effect, indicating that the chosen architectures already balanced complexity and generalization well. Therefore, for this CKD dataset, allocating more neurons (and layers) provided the greatest performance gain.